In [ ]:
import os
import sys

sys.path.append(os.path.abspath('../ciphers'))

import simon3264 as simon3264

cipher_dict = {
    "simon3264":simon3264
}

from DataGenerator import DataGenerator

import numpy as np
from os import urandom
from tensorflow.keras.regularizers import l2
from tensorflow.keras.backend import concatenate
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Dense, AveragePooling1D, GlobalAveragePooling1D, Conv1D, MaxPooling1D, Input, Reshape, Permute, Add, Flatten, BatchNormalization, Activation, Multiply
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler, CSVLogger
from pickle import dump
import tensorflow as tf
import tensorflow

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

def cyclic_lr(num_epochs, high_lr, low_lr):
    res = lambda i: low_lr + ((num_epochs-1) - i % num_epochs)/(num_epochs-1) * (high_lr - low_lr)
    return res

def make_checkpoint(datei):
    res = ModelCheckpoint(datei, monitor='val_loss', save_best_only = True)
    return res

bs = 4096

2025-09-13 16:31:27.270866: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def make_resnet(num_blocks=2, num_filters=32, num_outputs=1, ds=[64, 64], word_size=64, ks=3, depth=5, reg_param=0.0001, final_activation='sigmoid'):
    inp = Input(shape=(num_blocks * word_size * 2,))
    rs = Reshape((2 * num_blocks, word_size))(inp)
    perm = Permute((2,1))(rs)
    conv0 = Conv1D(num_filters, kernel_size=1, padding='same', kernel_regularizer=l2(reg_param))(perm)
    conv0 = BatchNormalization()(conv0)
    conv0 = Activation('relu')(conv0)
    shortcut = conv0
    for i in range(depth):
        conv1 = Conv1D(num_filters, kernel_size=ks, padding='same', kernel_regularizer=l2(reg_param))(shortcut)
        conv1 = BatchNormalization()(conv1)
        conv1 = Activation('relu')(conv1)
        conv2 = Conv1D(num_filters, kernel_size=ks, padding='same',kernel_regularizer=l2(reg_param))(conv1)
        conv2 = BatchNormalization()(conv2)
        conv2 = Activation('relu')(conv2)
        shortcut = Add()([shortcut, conv2])
    flat = Flatten()(shortcut)
    for d in ds:
        dense = Dense(d,kernel_regularizer=l2(reg_param))(flat)
        dense = BatchNormalization()(dense)
        dense = Activation('relu')(dense)
    out = Dense(num_outputs, activation=final_activation, kernel_regularizer=l2(reg_param))(dense)
    model = Model(inputs=inp, outputs=out)
    return model

In [ ]:
def validate(index, cipher, num_epochs, num_rounds=5, num_blocks=2, num_dataset=10**7, diff0=(0x0040,0), diff1=(0x0040,0), num_filters=32, ds=[64,64], ks=3, depth=1, reg_param=10**-5, loss='mse', wdir="./freshly_trained_nets/"):
    net = make_resnet(num_blocks=num_blocks, num_filters=num_filters, ds=ds, word_size=cipher.WORD_SIZE(), ks=ks, depth=depth, reg_param=reg_param)
    net.compile(optimizer='adam', loss=loss, metrics=['acc'])
        
    training_generator = DataGenerator(cipher, num_dataset, bs, num_rounds, diff0, diff1)
    validation_generator = DataGenerator(cipher, num_dataset//10, bs, num_rounds, diff0, diff1)
    
    check = make_checkpoint(wdir+'best'+str(num_rounds)+'depth'+str(depth)+'_'+str(index)+'.h5')
    lr = LearningRateScheduler(cyclic_lr(10,0.002, 0.0001))
    log_path = wdir + 'best' + str(num_rounds) + 'depth' + str(depth) + '_' + str(index) + '.log'
    csv_logger = CSVLogger(log_path, append=True)
    
    h = net.fit(training_generator, epochs=num_epochs, validation_data=validation_generator, callbacks=[lr, check, csv_logger])
    
    print("Best validation accuracy: ", np.max(h.history['val_acc']))
    with open(log_path, 'a') as log_file: 
        log_file.write(f"Best validation accuracy: {np.max(h.history['val_acc'])}\n")

    test_generator = DataGenerator(cipher, num_dataset//10, bs, num_rounds, diff0, diff1)
    
    net.load_weights(wdir+'best'+str(num_rounds)+'depth'+str(depth)+'_'+str(index)+'.h5')
    test_loss, test_acc = net.evaluate(test_generator)
    print("Test accuracy: ", test_acc)
    with open(log_path, 'a') as log_file:
        log_file.write(f"Test accuracy: {test_acc}\n")

    return net, h

In [ ]:
cipher = cipher_dict['simon3264']
diff0=(0x0000,0x0001)
diff1=(0x0000,0x0080)
validate(index=0, cipher=cipher_dict['simon3264'], num_epochs=20, num_rounds=6, num_blocks=2, num_dataset=10**7, diff0=diff0, diff1=diff1, num_filters=32, ds=[64,64], ks=3, depth=10, reg_param=10**-5, loss='mse', wdir="./simon3264/00000001_00000080/")

Epoch 1/20
2441/2441 [==============================] - 82s 30ms/step - loss: 0.0028 - acc: 0.9992 - val_loss: 0.2977 - val_acc: 0.5201 - lr: 0.0020
Epoch 2/20
2441/2441 [==============================] - 65s 27ms/step - loss: 5.6863e-05 - acc: 1.0000 - val_loss: 2.4860e-04 - val_acc: 1.0000 - lr: 0.0018
Epoch 3/20
2441/2441 [==============================] - 60s 25ms/step - loss: 0.0012 - acc: 0.9993 - val_loss: 0.0012 - val_acc: 1.0000 - lr: 0.0016
Epoch 4/20
2441/2441 [==============================] - 62s 25ms/step - loss: 8.2459e-04 - acc: 1.0000 - val_loss: 5.3064e-04 - val_acc: 1.0000 - lr: 0.0014
Epoch 5/20
2441/2441 [==============================] - 60s 24ms/step - loss: 3.3684e-04 - acc: 1.0000 - val_loss: 2.1722e-04 - val_acc: 1.0000 - lr: 0.0012
Epoch 6/20
2441/2441 [==============================] - 60s 25ms/step - loss: 1.1006e-04 - acc: 1.0000 - val_loss: 5.3748e-05 - val_acc: 1.0000 - lr: 9.4444e-04
Epoch 7/20
2441/2441 [==============================] - 62s 25ms/step 

(<keras.engine.functional.Functional at 0x7f1bd84edd30>,
 <keras.callbacks.History at 0x7f1bd8564c10>)

In [ ]:
cipher = cipher_dict['simon3264']
diff0=(0x0000,0x0020)
diff1=(0x0000,0x1000)
validate(index=0, cipher=cipher_dict['simon3264'], num_epochs=20, num_rounds=6, num_blocks=2, num_dataset=10**7, diff0=diff0, diff1=diff1, num_filters=32, ds=[64,64], ks=3, depth=10, reg_param=10**-5, loss='mse', wdir="./simon3264/00000020_00001000/")

Epoch 1/20
2441/2441 [==============================] - 44s 17ms/step - loss: 0.0030 - acc: 0.9991 - val_loss: 0.0026 - val_acc: 1.0000 - lr: 0.0020
Epoch 2/20
2441/2441 [==============================] - 48s 20ms/step - loss: 0.0018 - acc: 0.9996 - val_loss: 0.0011 - val_acc: 1.0000 - lr: 0.0018
Epoch 3/20
2441/2441 [==============================] - 51s 21ms/step - loss: 5.4275e-04 - acc: 1.0000 - val_loss: 1.9126e-04 - val_acc: 1.0000 - lr: 0.0016
Epoch 4/20
2441/2441 [==============================] - 67s 27ms/step - loss: 7.7479e-05 - acc: 1.0000 - val_loss: 7.9830e-05 - val_acc: 1.0000 - lr: 0.0014
Epoch 5/20
2441/2441 [==============================] - 59s 24ms/step - loss: 8.6520e-04 - acc: 0.9994 - val_loss: 0.0010 - val_acc: 1.0000 - lr: 0.0012
Epoch 6/20
2441/2441 [==============================] - 62s 25ms/step - loss: 6.7585e-04 - acc: 1.0000 - val_loss: 3.9517e-04 - val_acc: 1.0000 - lr: 9.4444e-04
Epoch 7/20
2441/2441 [==============================] - 59s 24ms/step - lo

(<keras.engine.functional.Functional at 0x7f1b703745e0>,
 <keras.callbacks.History at 0x7f1b702ec340>)

In [ ]:
cipher = cipher_dict['simon3264']
diff0=(0x0000,0x4000)
diff1=(0x0004,0x0000)
validate(index=0, cipher=cipher_dict['simon3264'], num_epochs=20, num_rounds=6, num_blocks=2, num_dataset=10**7, diff0=diff0, diff1=diff1, num_filters=32, ds=[64,64], ks=3, depth=10, reg_param=10**-5, loss='mse', wdir="./simon3264/00004000_00040000/")

Epoch 1/20
2441/2441 [==============================] - 68s 25ms/step - loss: 0.0349 - acc: 0.9654 - val_loss: 0.0339 - val_acc: 0.9661 - lr: 0.0020
Epoch 2/20
2441/2441 [==============================] - 65s 27ms/step - loss: 0.0227 - acc: 0.9776 - val_loss: 0.0220 - val_acc: 0.9781 - lr: 0.0018
Epoch 3/20
2441/2441 [==============================] - 61s 25ms/step - loss: 0.0217 - acc: 0.9782 - val_loss: 0.0212 - val_acc: 0.9787 - lr: 0.0016
Epoch 4/20
2441/2441 [==============================] - 61s 25ms/step - loss: 0.0207 - acc: 0.9791 - val_loss: 0.0206 - val_acc: 0.9793 - lr: 0.0014
Epoch 5/20
2441/2441 [==============================] - 62s 25ms/step - loss: 0.0201 - acc: 0.9798 - val_loss: 0.0198 - val_acc: 0.9801 - lr: 0.0012
Epoch 6/20
2441/2441 [==============================] - 67s 28ms/step - loss: 0.0196 - acc: 0.9802 - val_loss: 0.0196 - val_acc: 0.9804 - lr: 9.4444e-04
Epoch 7/20
2441/2441 [==============================] - 61s 25ms/step - loss: 0.0192 - acc: 0.9806 - v

(<keras.engine.functional.Functional at 0x7f1aa8221100>,
 <keras.callbacks.History at 0x7f1aa82a5160>)

In [ ]:
cipher = cipher_dict['simon3264']
diff0=(0x0020,0x0000)
diff1=(0x0200,0x0000)
validate(index=0, cipher=cipher_dict['simon3264'], num_epochs=20, num_rounds=6, num_blocks=2, num_dataset=10**7, diff0=diff0, diff1=diff1, num_filters=32, ds=[64,64], ks=3, depth=10, reg_param=10**-5, loss='mse', wdir="./simon3264/00200000_02000000/")

Epoch 1/20
2441/2441 [==============================] - 68s 25ms/step - loss: 0.1049 - acc: 0.8588 - val_loss: 0.0746 - val_acc: 0.9092 - lr: 0.0020
Epoch 2/20
2441/2441 [==============================] - 59s 24ms/step - loss: 0.0686 - acc: 0.9158 - val_loss: 0.0686 - val_acc: 0.9145 - lr: 0.0018
Epoch 3/20
2441/2441 [==============================] - 63s 26ms/step - loss: 0.0634 - acc: 0.9225 - val_loss: 0.0632 - val_acc: 0.9242 - lr: 0.0016
Epoch 4/20
2441/2441 [==============================] - 61s 25ms/step - loss: 0.0595 - acc: 0.9287 - val_loss: 0.0596 - val_acc: 0.9285 - lr: 0.0014
Epoch 5/20
2441/2441 [==============================] - 62s 26ms/step - loss: 0.0563 - acc: 0.9340 - val_loss: 0.0561 - val_acc: 0.9350 - lr: 0.0012
Epoch 6/20
2441/2441 [==============================] - 63s 26ms/step - loss: 0.0534 - acc: 0.9379 - val_loss: 0.0526 - val_acc: 0.9388 - lr: 9.4444e-04
Epoch 7/20
2441/2441 [==============================] - 61s 25ms/step - loss: 0.0511 - acc: 0.9406 - v

(<keras.engine.functional.Functional at 0x7f1a68414c40>,
 <keras.callbacks.History at 0x7f1a68427e80>)

In [ ]:
cipher = cipher_dict['simon3264']
diff0=(0x0800,0x0000)
diff1=(0x1000,0x0000)
validate(index=0, cipher=cipher_dict['simon3264'], num_epochs=20, num_rounds=6, num_blocks=2, num_dataset=10**7, diff0=diff0, diff1=diff1, num_filters=32, ds=[64,64], ks=3, depth=10, reg_param=10**-5, loss='mse', wdir="./simon3264/08000000_10000000/")

Epoch 1/20
2441/2441 [==============================] - 67s 25ms/step - loss: 0.0712 - acc: 0.9095 - val_loss: 0.0507 - val_acc: 0.9383 - lr: 0.0020
Epoch 2/20
2441/2441 [==============================] - 61s 25ms/step - loss: 0.0447 - acc: 0.9464 - val_loss: 0.0393 - val_acc: 0.9551 - lr: 0.0018
Epoch 3/20
2441/2441 [==============================] - 60s 24ms/step - loss: 0.0365 - acc: 0.9582 - val_loss: 0.0391 - val_acc: 0.9540 - lr: 0.0016
Epoch 4/20
2441/2441 [==============================] - 63s 26ms/step - loss: 0.0335 - acc: 0.9620 - val_loss: 0.0368 - val_acc: 0.9592 - lr: 0.0014
Epoch 5/20
2441/2441 [==============================] - 65s 27ms/step - loss: 0.0308 - acc: 0.9652 - val_loss: 0.0302 - val_acc: 0.9655 - lr: 0.0012
Epoch 6/20
2441/2441 [==============================] - 61s 25ms/step - loss: 0.0287 - acc: 0.9674 - val_loss: 0.0283 - val_acc: 0.9679 - lr: 9.4444e-04
Epoch 7/20
2441/2441 [==============================] - 61s 25ms/step - loss: 0.0270 - acc: 0.9693 - v

(<keras.engine.functional.Functional at 0x7f1a28606340>,
 <keras.callbacks.History at 0x7f1a286b2ca0>)